In [2]:
import pandas as pd

df = pd.read_csv('../data/dirty_cafe_sales.csv')
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [3]:
# Reemplazamos los valores "sucios" por NaN real de pandas
df = df.replace(['ERROR', 'UNKNOWN'], pd.NA)

# Verificamos que ya no existan
for col in df.select_dtypes(include=['object', 'str']).columns:
    print(col, ':', df[col].unique())

Transaction ID : <StringArray>
['TXN_1961373', 'TXN_4977031', 'TXN_4271903', 'TXN_7034554', 'TXN_3160411',
 'TXN_2602893', 'TXN_4433211', 'TXN_6699534', 'TXN_4717867', 'TXN_2064365',
 ...
 'TXN_1538510', 'TXN_3897619', 'TXN_2739140', 'TXN_4766549', 'TXN_7851634',
 'TXN_7672686', 'TXN_9659401', 'TXN_5255387', 'TXN_7695629', 'TXN_6170729']
Length: 10000, dtype: str
Item : <StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',        nan,
 'Sandwich',    'Juice',      'Tea']
Length: 9, dtype: str
Quantity : <StringArray>
['2', '4', '5', '3', '1', nan]
Length: 6, dtype: str
Price Per Unit : <StringArray>
['2.0', '3.0', '1.0', '5.0', '4.0', '1.5', nan]
Length: 7, dtype: str
Total Spent : <StringArray>
[ '4.0', '12.0',    nan, '10.0', '20.0',  '9.0', '16.0', '15.0', '25.0',
  '8.0',  '5.0',  '3.0',  '6.0',  '2.0',  '1.0',  '7.5',  '4.5',  '1.5']
Length: 18, dtype: str
Payment Method : <StringArray>
['Credit Card', 'Cash', nan, 'Digital Wallet']
Length: 4, dtype: str
Locat

In [4]:
df.isnull().sum()

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

In [5]:
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')
df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce')

In [6]:
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  str           
 1   Item              9031 non-null   str           
 2   Quantity          9521 non-null   float64       
 3   Price Per Unit    9467 non-null   float64       
 4   Total Spent       9498 non-null   float64       
 5   Payment Method    6822 non-null   str           
 6   Location          6039 non-null   str           
 7   Transaction Date  9540 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(3), str(4)
memory usage: 625.1 KB


In [8]:
df.isnull().sum()

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

In [9]:
# Calculamos el valor "correcto" según la fórmula
calculado = df['Quantity'] * df['Price Per Unit']

# Solo llenamos los NaN de Total Spent con el valor calculado
# (si Total Spent ya tenía un valor, lo dejamos como estaba)
df['Total Spent'] = df['Total Spent'].fillna(calculado)

In [10]:
df['Total Spent'].isnull().sum()

np.int64(40)

In [11]:
# Price Per Unit = Total Spent / Quantity
calculado_precio = df['Total Spent'] / df['Quantity']
df['Price Per Unit'] = df['Price Per Unit'].fillna(calculado_precio)

In [12]:
# Quantity = Total Spent / Price Per Unit
calculado_cantidad = df['Total Spent'] / df['Price Per Unit']
df['Quantity'] = df['Quantity'].fillna(calculado_cantidad)

In [13]:
df[['Quantity', 'Price Per Unit', 'Total Spent']].isnull().sum()

Quantity          38
Price Per Unit    38
Total Spent       40
dtype: int64

In [14]:
# Vemos si hay valores no enteros en Quantity
df[df['Quantity'] % 1 != 0][['Quantity', 'Price Per Unit', 'Total Spent']]

,Quantity,Price Per Unit,Total Spent
236,NaN,5.0,NaN
278,NaN,3.0,NaN
629,NaN,NaN,12.0
641,NaN,3.0,NaN
738,NaN,4.0,NaN
912,NaN,NaN,20.0
1008,NaN,NaN,3.0
1436,NaN,NaN,6.0
1482,NaN,NaN,16.0
2330,NaN,NaN,5.0


In [15]:
# Excluimos los NaN explícitamente, y buscamos solo decimales reales
df[df['Quantity'].notna() & (df['Quantity'] % 1 != 0)][['Quantity', 'Price Per Unit', 'Total Spent']]

,Quantity,Price Per Unit,Total Spent
